# Agent dùng model đã train (Ollama, so với model gốc)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hytmk2912/Huyen/blob/main/notebooks/agent_trained_colab.ipynb)

Notebook này lấy adapter bạn đã train bằng `train_colab` (repo riêng tư `<tên>/huyen-<model>-qlora`), gộp vào model gốc, xuất file GGUF, tạo 2 model trong Ollama rồi cho agent của repo làm 6 nhiệm vụ mẫu **với từng model** và in bảng so sánh:
- **model gốc**: model gốc cùng revision, cùng cách xuất (GGUF `q8_0`), không có adapter;
- **model đã train**: model gốc + adapter đã gộp.

Hai model đi qua đúng cùng một đường (cùng revision, dtype, kiểu lượng tử, chat template), nên chênh lệch là do train, không do cách xuất.

**Cần chuẩn bị:** đã chạy `train_colab` tới Bước 11 (đẩy adapter) với đúng model sẽ chọn ở Bước 1; token Hugging Face trong Colab Secrets tên `HF_TOKEN` (bật Notebook access); GPU T4 (menu Runtime → Change runtime type).

**Cách chạy:** chọn model ở Bước 1 rồi Runtime → Run all. Thời gian (ước lượng, chưa đo trên Colab thật): `smoke` khoảng 15–20 phút; `light` khoảng 45–70 phút (tải model gốc 8 GB, xuất 2 file GGUF khoảng 4,3 GB mỗi file, rồi 2 lượt agent).

In [ ]:
# Bước 1: chọn model đã train bằng notebook train_colab (smoke = Qwen2.5-0.5B, light = Qwen3-4B), rồi mới bấm Run all.
MODEL = "smoke"  # @param ["smoke", "light"]
print("Đã chọn model:", MODEL, "| adapter: .runs/colab_" + MODEL + "/adapter | model Ollama: huyen-" + MODEL + "-goc và huyen-" + MODEL + "-da-train")

In [ ]:
# Bước 2: kiểm tra GPU. Gộp adapter và chạy Ollama cần GPU T4 (khoảng 15 GB); trên CPU sẽ rất chậm.
# Nếu báo "Chưa có GPU": menu Runtime → Change runtime type → chọn "T4 GPU" → Save, rồi Run all lại.
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Vào menu Runtime → Change runtime type → chọn T4 GPU → Save, rồi bấm Run all lại.")
print("GPU:", torch.cuda.get_device_name(0), "| bộ nhớ:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Bước 3: tải code của repo, cài thư viện đã ghim (không cài lại torch), tải llama.cpp bản b11205 (chỉ dùng công cụ chuyển sang GGUF)
# và cài Ollama bản 0.34.4. Chạy lại notebook thì không tải lại những gì đã có. Từ đây, lệnh nào lỗi thì Run all dừng ở ô đó.
import os

if not os.path.isdir("/content/Huyen/local_ai"):
    !git clone --depth 1 https://github.com/hytmk2912/Huyen.git /content/Huyen
%cd /content/Huyen
if not os.path.isdir("local_ai"):
    raise RuntimeError("Tải code thất bại: chưa có thư mục /content/Huyen/local_ai. Kiểm tra mạng rồi chạy lại ô này.")
from local_ai.colab import run  # chạy lệnh: lệnh lỗi thì ô báo đỏ và Run all dừng ở đúng chỗ lỗi

run("git pull --ff-only", "Bước 3 (cập nhật code)")
run("pip install -q transformers==5.17.0 peft==0.21.0 accelerate==1.15.0 sentencepiece==0.2.2", "Bước 3 (cài thư viện)")
# Colab cài sẵn torchao; bản đó không hợp với các thư viện đã ghim (gặp khi train ngày 25/9). Repo không dùng torchao nên gỡ đi.
run("pip uninstall -y -q torchao", "Bước 3 (gỡ torchao)")
if not os.path.isfile("/content/llama.cpp/convert_hf_to_gguf.py"):
    run(["git", "clone", "--depth", "1", "--branch", "b11205", "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], "Bước 3 (tải llama.cpp)")
run("apt-get -qq install -y zstd pciutils", "Bước 3 (cài zstd)", quiet=True)
run("curl -fsSL -o /tmp/ollama_install.sh https://ollama.com/install.sh", "Bước 3 (tải bản cài Ollama)")
run(["sh", "/tmp/ollama_install.sh"], "Bước 3 (cài Ollama)", env={"OLLAMA_VERSION": "0.34.4"})

In [ ]:
# Bước 4: lấy HF_TOKEN từ Colab Secrets (biểu tượng chìa khóa 🔑) để tải adapter từ repo riêng tư. Token không bị in ra và không lưu vào notebook.
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as error:
    raise RuntimeError("Chưa đọc được HF_TOKEN: hãy thêm secret tên HF_TOKEN và bật Notebook access (" + type(error).__name__ + ")") from None

from huggingface_hub import whoami

HF_USER = whoami()["name"]
HUB_REPO = f"{HF_USER}/huyen-{MODEL}-qlora"  # repo riêng tư mà train_colab đã đẩy adapter lên (Bước 11)
print("Tài khoản Hugging Face:", HF_USER, "| repo:", HUB_REPO)

In [ ]:
# Bước 5: tải adapter đã train (chỉ các file adapter, vài chục MB) về đúng thư mục adapter_path của mục <model>-colab trong configs/models/platform.json.
run(f"python -m local_ai.training.hub pull-adapter --repo {HUB_REPO} --adapter-dir .runs/colab_{MODEL}/adapter", "Bước 5 (tải adapter)")

In [ ]:
# Bước 6: xuất 2 file GGUF (q8_0) cùng một cách: model gốc (--no-adapter) và model gốc + adapter đã gộp.
# Mỗi lần: nạp model gốc không nén theo revision đã ghim → (gộp adapter) → safetensors → GGUF bằng llama.cpp → Modelfile cho Ollama.
# Chạy lại notebook thì bỏ qua file đã xuất xong (có export_info.json).
import os

for name, extra in (("goc", "--no-adapter"), ("da-train", "")):
    if os.path.isfile(f".runs/gguf/{MODEL}-{name}/export_info.json"):
        print("Đã có", f".runs/gguf/{MODEL}-{name}/model.gguf", "- bỏ qua")
        continue
    run(f"python -m local_ai.training.export --model {MODEL}-colab {extra} --output .runs/gguf/{MODEL}-{name} --llama-cpp /content/llama.cpp", f"Bước 6 (xuất {name})")

In [ ]:
# Bước 7: chạy máy chủ Ollama ở nền (log ghi vào /content/ollama.log) và đợi tới khi nó trả lời.
import subprocess
import time
import urllib.request


def ollama_ready():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        return True
    except OSError:
        return False


if not ollama_ready():
    subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    if ollama_ready():
        print("Ollama đã chạy.")
        break
    time.sleep(1)
else:
    raise RuntimeError("Ollama chưa chạy sau 60 giây; hãy xem file /content/ollama.log")

In [ ]:
# Bước 8: tạo 2 model trong Ollama từ file GGUF và Modelfile vừa xuất (chat template và tham số giống model cùng họ trong thư viện Ollama).
run(f"ollama create huyen-{MODEL}-goc -f .runs/gguf/{MODEL}-goc/Modelfile", "Bước 8 (tạo model gốc)")
run(f"ollama create huyen-{MODEL}-da-train -f .runs/gguf/{MODEL}-da-train/Modelfile", "Bước 8 (tạo model đã train)")
run("ollama list", "Bước 8 (danh sách model)")

In [ ]:
# Bước 9: agent làm 6 nhiệm vụ mẫu với model gốc (mục ollama-<model>-goc). Mỗi nhiệm vụ in trace; dòng cuối là tỉ lệ thành công.
run(f"python -m local_ai.agents.tasks --model ollama-{MODEL}-goc --workspace .runs/agent_tasks/{MODEL}-goc --output .runs/agent_tasks/{MODEL}-goc.json", "Bước 9 (agent với model gốc)")

In [ ]:
# Bước 10: agent làm đúng 6 nhiệm vụ đó với model đã train (mục ollama-<model>-da-train).
run(f"python -m local_ai.agents.tasks --model ollama-{MODEL}-da-train --workspace .runs/agent_tasks/{MODEL}-da-train --output .runs/agent_tasks/{MODEL}-da-train.json", "Bước 10 (agent với model đã train)")

In [ ]:
# Bước 11: bảng so sánh từng nhiệm vụ, tỉ lệ thành công, thời gian, nhiệm vụ mới đạt / mới trượt sau khi train.
run(f"python -m local_ai.agents.compare .runs/agent_tasks/{MODEL}-goc.json .runs/agent_tasks/{MODEL}-da-train.json --labels 'model gốc' 'model đã train' --output .runs/agent_tasks/{MODEL}-so-sanh.json", "Bước 11 (so sánh)")

## Kết quả nằm ở đâu
- **Bảng so sánh:** in dưới Bước 11 (chi tiết JSON ở `.runs/agent_tasks/<model>-so-sanh.json`). Trace từng nhiệm vụ in dưới Bước 9 và 10. Các file này mất khi Colab tắt, nên hãy chụp màn hình Bước 11 gửi lại.
- Một nhiệm vụ chỉ tính là **đạt** khi câu trả lời đúng **và** agent đã thật sự gọi công cụ cần dùng.
- Agent gửi `temperature` 0 và `seed` cố định, nên chạy lại ra cùng kết quả; chênh lệch giữa 2 model là do train.

## Lỗi hay gặp
- **Bước 5 báo "chưa có adapter" hoặc "không thấy repo":** chưa chạy `train_colab` tới Bước 11 với model này, hoặc chọn nhầm model ở Bước 1.
- **Bước 6 báo thiếu sentencepiece:** chạy lại Bước 3.
- **Bước 6 hết bộ nhớ khi gộp `light`:** Runtime → Restart session rồi Run all (file đã xuất xong được giữ lại).
- **"Không kết nối được server model":** Ollama chưa chạy; chạy lại Bước 7 (xem `/content/ollama.log`).
- **Ô báo đỏ "Bước N lỗi (mã thoát ...)":** đọc thông báo ngay phía trên dòng đỏ, sửa xong thì chạy lại từ ô đó.